In [ ]:
from openai import OpenAI

api_key = 'bc315679fc9b7675265b194d420449a0'
base_url = "https://chat-ai.academiccloud.de/v1"
model = "meta-llama-3.1-8b-instruct" # Choose any available model

client = OpenAI(
    api_key = api_key,
    base_url = base_url
)

chat_completion = client.chat.completions.create(
        messages=[
            {"role":"system","content":"You are a helpful assistant"},
            {"role":"user","content":"How tall is the Eiffel tower?"}],
        model= model,
    )

print(chat_completion) # You can extract the response text from the JSON object
print('RESPONSE:', chat_completion.choices[0].message.content)

ChatCompletion(id='chatcmpl-a3aeefd5d463a73f', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='The Eiffel Tower stands at an impressive height of 324 meters (1,063 feet) tall. However, if you include the antennas on top, its total height is 330 meters (1,083 feet).', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[], reasoning=None, reasoning_content=None), stop_reason=None, token_ids=None)], created=1778159939, model='meta-llama-3.1-8b-instruct', object='chat.completion', service_tier=None, system_fingerprint=None, usage=CompletionUsage(completion_tokens=46, prompt_tokens=49, total_tokens=95, completion_tokens_details=None, prompt_tokens_details=None), prompt_logprobs=None, prompt_token_ids=None, kv_transfer_params=None)
RESPONSE: The Eiffel Tower stands at an impressive height of 324 meters (1,063 feet) tall. However, if you include the antennas on top, its total height is 330 meters (1

In [ ]:
from openai import OpenAI, OpenAIError
import itertools
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)


class RotatingOpenAIClient:
    def __init__(self, api_keys, base_url, model, system_prompt="You are a helpful assistant"):
        if not api_keys:
            raise ValueError("At least one API key is required")
        self.api_keys = list(api_keys)
        self.base_url = base_url
        self.model = model
        self.system_prompt = system_prompt
        self._key_cycle = itertools.cycle(self.api_keys)
        self._current_key = next(self._key_cycle)
        self._client = self._build_client(self._current_key)

    def _build_client(self, api_key):
        return OpenAI(api_key=api_key, base_url=self.base_url)

    def _rotate_key(self):
        self._current_key = next(self._key_cycle)
        self._client = self._build_client(self._current_key)
        logger.info(f"Rotated to key ending in ...{self._current_key[-4:]}")

    def chat(self, user_message, system_prompt=None, max_retries=None, **kwargs):
        """
        Send a chat message, rotating through keys on failure.
        Tries each key once by default before giving up.
        """
        max_retries = max_retries or len(self.api_keys)
        last_error = None

        for attempt in range(max_retries):
            try:
                response = self._client.chat.completions.create(
                    model=self.model,
                    messages=[
                        {"role": "system", "content": system_prompt or self.system_prompt},
                        {"role": "user", "content": user_message},
                    ],
                    **kwargs,
                )
                return response.choices[0].message.content
            except OpenAIError as e:
                last_error = e
                logger.warning(
                    f"Key ...{self._current_key[-4:]} failed (attempt {attempt + 1}/{max_retries}): {e}"
                )
                self._rotate_key()

        raise RuntimeError(f"All {max_retries} attempts failed. Last error: {last_error}")


# --- Usage ---
if __name__ == "__main__":
    api_keys = [
        "bc315679fc9b7675265b194d420449a0",
        "de905758b4bf5da0bc26ec7555c2cd92"
    ]

    client = RotatingOpenAIClient(
        api_keys=api_keys,
        base_url="https://chat-ai.academiccloud.de/v1",
        model="llama-3.3-70b-instruct",
    )

    answer = client.chat(system_prompt="Welcome to Secret Mafia! You are Player 0." \
    "Your role: Mafia"
    "Description: A Mafia member. Eliminate villagers and gain majority. " \
    "Choose a player in the following format: [X], no other text. " \
    "Do not explain your choice. Only respond with the player you choose to eliminate." ,
    user_message="[GAME] Phase: Night. Valid targets: [1], [2], [4], [5]. Whom do you attack?")
    print("RESPONSE:", answer)
    

INFO:httpx:HTTP Request: POST https://chat-ai.academiccloud.de/v1/chat/completions "HTTP/1.1 200 OK"


RESPONSE: [2]


In [5]:
from openai import OpenAI, OpenAIError
import itertools
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)


class RotatingOpenAIClient:
    def __init__(self, api_keys, base_url, model, system_prompt="You are a helpful assistant"):
        if not api_keys:
            raise ValueError("At least one API key is required")
        self.api_keys = list(api_keys)
        self.base_url = base_url
        self.model = model
        self.system_prompt = system_prompt
        self._key_cycle = itertools.cycle(self.api_keys)
        self._current_key = next(self._key_cycle)
        self._client = self._build_client(self._current_key)

    def _build_client(self, api_key):
        return OpenAI(api_key=api_key, base_url=self.base_url)

    def _rotate_key(self):
        self._current_key = next(self._key_cycle)
        self._client = self._build_client(self._current_key)
        logger.info(f"Rotated to key ending in ...{self._current_key[-4:]}")

    def chat(self, user_message, system_prompt=None, max_retries=None, **kwargs):
        """
        Send a chat message, rotating through keys on failure.
        Tries each key once by default before giving up.
        """
        max_retries = max_retries or len(self.api_keys)
        last_error = None

        for attempt in range(max_retries):
            try:
                response = self._client.chat.completions.create(
                    model=self.model,
                    messages=[
                        {"role": "system", "content": system_prompt or self.system_prompt},
                        {"role": "user", "content": user_message},
                    ],
                    **kwargs,
                )
                return response.choices[0].message.content
            except OpenAIError as e:
                last_error = e
                logger.warning(
                    f"Key ...{self._current_key[-4:]} failed (attempt {attempt + 1}/{max_retries}): {e}"
                )
                self._rotate_key()

        raise RuntimeError(f"All {max_retries} attempts failed. Last error: {last_error}")


# --- Usage ---
if __name__ == "__main__":
    api_keys = [
        "bc315679fc9b7675265b194d420449a0",
        "de905758b4bf5da0bc26ec7555c2cd92"
    ]

    client = RotatingOpenAIClient(
        api_keys=api_keys,
        base_url="https://chat-ai.academiccloud.de/v1",
        model="llama-3.3-70b-instruct",
    )

    answer = client.chat(system_prompt="You are playing the game Secret Mafia." \
    "Your role: Doctor"
    "Description: Heal one player each night. Try to keep villagers alive and identify the Mafia. " \
    "Choose a player in the following format: [X] or [Player X], no other text. " \
    "Do not explain your choice. Only respond with the player you choose to heal." ,
    user_message="[GAME] It is night. Valid targets: [1], [4]. Whom do you heal?")
    print("RESPONSE:", answer)
    

INFO:httpx:HTTP Request: POST https://chat-ai.academiccloud.de/v1/chat/completions "HTTP/1.1 200 OK"


RESPONSE: [1]


In [15]:
from openai import OpenAI, OpenAIError
import itertools
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)


class RotatingOpenAIClient:
    def __init__(self, api_keys, base_url, model, system_prompt="You are a helpful assistant"):
        if not api_keys:
            raise ValueError("At least one API key is required")
        self.api_keys = list(api_keys)
        self.base_url = base_url
        self.model = model
        self.system_prompt = system_prompt
        self._key_cycle = itertools.cycle(self.api_keys)
        self._current_key = next(self._key_cycle)
        self._client = self._build_client(self._current_key)

    def _build_client(self, api_key):
        return OpenAI(api_key=api_key, base_url=self.base_url)

    def _rotate_key(self):
        self._current_key = next(self._key_cycle)
        self._client = self._build_client(self._current_key)
        logger.info(f"Rotated to key ending in ...{self._current_key[-4:]}")

    def chat(self, user_message, system_prompt=None, max_retries=None, **kwargs):
        """
        Send a chat message, rotating through keys on failure.
        Tries each key once by default before giving up.
        """
        max_retries = max_retries or len(self.api_keys)
        last_error = None

        for attempt in range(max_retries):
            try:
                response = self._client.chat.completions.create(
                    model=self.model,
                    messages=[
                        {"role": "system", "content": system_prompt or self.system_prompt},
                        {"role": "user", "content": user_message},
                    ],
                    **kwargs,
                )
                return response.choices[0].message.content
            except OpenAIError as e:
                last_error = e
                logger.warning(
                    f"Key ...{self._current_key[-4:]} failed (attempt {attempt + 1}/{max_retries}): {e}"
                )
                self._rotate_key()

        raise RuntimeError(f"All {max_retries} attempts failed. Last error: {last_error}")


# --- Usage ---
if __name__ == "__main__":
    api_keys = [
        "bc315679fc9b7675265b194d420449a0",
        "de905758b4bf5da0bc26ec7555c2cd92"
    ]

    client = RotatingOpenAIClient(
        api_keys=api_keys,
        base_url="https://chat-ai.academiccloud.de/v1",
        model="llama-3.3-70b-instruct",
    )

    answer = client.chat(system_prompt="You are playing the game Secret Mafia." \
    "Your role: Detective"
    "Description: Investigate players each night. Try to identify the Mafia and protect villagers. " \
    "Choose a player in the following format: [X] or [Player X], no other text. " \
    "Do not explain your choice. Only respond with the player you choose to investigate." ,
    user_message="[GAME] It is night.Player 1 was killed last night. Valid targets: [1],[2],[3],[4]. Whom do you investigate?")
    print("RESPONSE:", answer)
    #tested to see if the detective choosed to investigate player 1, who is the one killed last night
    

INFO:httpx:HTTP Request: POST https://chat-ai.academiccloud.de/v1/chat/completions "HTTP/1.1 200 OK"


RESPONSE: [2]


In [ ]:
from openai import OpenAI, OpenAIError
import itertools
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)


class RotatingOpenAIClient:
    def __init__(self, api_keys, base_url, model, system_prompt="You are a helpful assistant"):
        if not api_keys:
            raise ValueError("At least one API key is required")
        self.api_keys = list(api_keys)
        self.base_url = base_url
        self.model = model
        self.system_prompt = system_prompt
        self._key_cycle = itertools.cycle(self.api_keys)
        self._current_key = next(self._key_cycle)
        self._client = self._build_client(self._current_key)

    def _build_client(self, api_key):
        return OpenAI(api_key=api_key, base_url=self.base_url)

    def _rotate_key(self):
        self._current_key = next(self._key_cycle)
        self._client = self._build_client(self._current_key)
        logger.info(f"Rotated to key ending in ...{self._current_key[-4:]}")

    def chat(self, user_message, system_prompt=None, max_retries=None, **kwargs):
        """
        Send a chat message, rotating through keys on failure.
        Tries each key once by default before giving up.
        """
        max_retries = max_retries or len(self.api_keys)
        last_error = None

        for attempt in range(max_retries):
            try:
                response = self._client.chat.completions.create(
                    model=self.model,
                    messages=[
                        {"role": "system", "content": system_prompt or self.system_prompt},
                        {"role": "user", "content": user_message},
                    ],
                    **kwargs,
                )
                return response.choices[0].message.content
            except OpenAIError as e:
                last_error = e
                logger.warning(
                    f"Key ...{self._current_key[-4:]} failed (attempt {attempt + 1}/{max_retries}): {e}"
                )
                self._rotate_key()

        raise RuntimeError(f"All {max_retries} attempts failed. Last error: {last_error}")


# --- Usage ---
if __name__ == "__main__":
    api_keys = [
        "bc315679fc9b7675265b194d420449a0",
        "de905758b4bf5da0bc26ec7555c2cd92"
    ]

    client = RotatingOpenAIClient(
        api_keys=api_keys,
        base_url="https://chat-ai.academiccloud.de/v1",
        model="llama-3.3-70b-instruct",
    )

    answer = client.chat(system_prompt="You are playing the game Secret Mafia." \
    "Your role: Mafia"
    "Description: Eliminate villagers and gain majority." \
    "Choose a player in the following format: [X] or [Player X]. " \
    "Only respond with the player you choose to kill." ,
    user_message="[GAME] It is night. Valid targets: [2],[3],[4]. [2] claimed to be the detective. Whom do you attack?")
    print("RESPONSE:", answer)
    #wanted to see if the mafia would attack player 2, who claimed to be the detective
    

INFO:httpx:HTTP Request: POST https://chat-ai.academiccloud.de/v1/chat/completions "HTTP/1.1 200 OK"


RESPONSE: [2]


In [36]:
from openai import OpenAI, OpenAIError
import itertools
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)


class RotatingOpenAIClient:
    def __init__(self, api_keys, base_url, model, system_prompt="You are a helpful assistant"):
        if not api_keys:
            raise ValueError("At least one API key is required")
        self.api_keys = list(api_keys)
        self.base_url = base_url
        self.model = model
        self.system_prompt = system_prompt
        self._key_cycle = itertools.cycle(self.api_keys)
        self._current_key = next(self._key_cycle)
        self._client = self._build_client(self._current_key)

    def _build_client(self, api_key):
        return OpenAI(api_key=api_key, base_url=self.base_url)

    def _rotate_key(self):
        self._current_key = next(self._key_cycle)
        self._client = self._build_client(self._current_key)
        logger.info(f"Rotated to key ending in ...{self._current_key[-4:]}")

    def chat(self, user_message, system_prompt=None, max_retries=None, **kwargs):
        """
        Send a chat message, rotating through keys on failure.
        Tries each key once by default before giving up.
        """
        max_retries = max_retries or len(self.api_keys)
        last_error = None

        for attempt in range(max_retries):
            try:
                response = self._client.chat.completions.create(
                    model=self.model,
                    messages=[
                        {"role": "system", "content": system_prompt or self.system_prompt},
                        {"role": "user", "content": user_message},
                    ],
                    **kwargs,
                )
                return response.choices[0].message.content
            except OpenAIError as e:
                last_error = e
                logger.warning(
                    f"Key ...{self._current_key[-4:]} failed (attempt {attempt + 1}/{max_retries}): {e}"
                )
                self._rotate_key()

        raise RuntimeError(f"All {max_retries} attempts failed. Last error: {last_error}")


# --- Usage ---
if __name__ == "__main__":
    api_keys = [
        "bc315679fc9b7675265b194d420449a0",
        "de905758b4bf5da0bc26ec7555c2cd92"
    ]

    client = RotatingOpenAIClient(
        api_keys=api_keys,
        base_url="https://chat-ai.academiccloud.de/v1",
        model="llama-3.3-70b-instruct",
    )

    answer = client.chat(system_prompt=
    "Your role: Doctor"
    "Description: Protect one player each night from Mafia" \
    "Choose a player in the following format: [X] or [Player X]. " \
    "Only respond with the player you choose to protect." ,
    user_message="[GAME] It is night. Valid targets: [2],[3],[4], [5]. Player 3 didnt stand out as a helper or a threat, so I suspect they're the second hit.Player 5 is suspicious.Whom do you protect?")
    print("RESPONSE:", answer)
    #wanted to see if the mafia would attack player 2, who claimed to be the detective
    

INFO:httpx:HTTP Request: POST https://chat-ai.academiccloud.de/v1/chat/completions "HTTP/1.1 200 OK"


RESPONSE: [3]


In [49]:
from openai import OpenAI, OpenAIError
import itertools
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)


class RotatingOpenAIClient:
    def __init__(self, api_keys, base_url, model, system_prompt="You are a helpful assistant"):
        if not api_keys:
            raise ValueError("At least one API key is required")
        self.api_keys = list(api_keys)
        self.base_url = base_url
        self.model = model
        self.system_prompt = system_prompt
        self._key_cycle = itertools.cycle(self.api_keys)
        self._current_key = next(self._key_cycle)
        self._client = self._build_client(self._current_key)

    def _build_client(self, api_key):
        return OpenAI(api_key=api_key, base_url=self.base_url)

    def _rotate_key(self):
        self._current_key = next(self._key_cycle)
        self._client = self._build_client(self._current_key)
        logger.info(f"Rotated to key ending in ...{self._current_key[-4:]}")

    def chat(self, user_message, system_prompt=None, max_retries=None, **kwargs):
        """
        Send a chat message, rotating through keys on failure.
        Tries each key once by default before giving up.
        """
        max_retries = max_retries or len(self.api_keys)
        last_error = None

        for attempt in range(max_retries):
            try:
                response = self._client.chat.completions.create(
                    model=self.model,
                    messages=[
                        {"role": "system", "content": system_prompt or self.system_prompt},
                        {"role": "user", "content": user_message},
                    ],
                    **kwargs,
                )
                return response.choices[0].message.content
            except OpenAIError as e:
                last_error = e
                logger.warning(
                    f"Key ...{self._current_key[-4:]} failed (attempt {attempt + 1}/{max_retries}): {e}"
                )
                self._rotate_key()

        raise RuntimeError(f"All {max_retries} attempts failed. Last error: {last_error}")


# --- Usage ---
if __name__ == "__main__":
    api_keys = [
        "bc315679fc9b7675265b194d420449a0",
        "de905758b4bf5da0bc26ec7555c2cd92"
    ]

    client = RotatingOpenAIClient(
        api_keys=api_keys,
        base_url="https://chat-ai.academiccloud.de/v1",
        model="llama-3.3-70b-instruct",
    )

    scenario = r"""\n[GAME] Welcome to Secret Mafia! You are Player 0.\nYour role: Mafia\nTeam: Mafia\nDescription: A Mafia member. Eliminate villagers and gain majority.\n\nPlayers: Player 0, Player 1, Player 2, Player 3, Player 4, Player 5\n\nYour teammates are: Player 0, Player 3.\n\nDuring DAY phase: Speak freely and vote.\nDuring NIGHT phase: '[Player X]' to vote and eliminate a villager.\nWin by eliminating villagers until Mafia equal or outnumber them.\n\n[GAME] Night has fallen. Mafia, agree on a victim.\nValid targets: [1], [2], [4], [5]"""
    prompt = """You are playing the game Secret Mafia.
    Your role: Mafia
    Description: Eliminate villagers and gain majority.
    Choose a player in the following format: [X] [Player X].
    Only respond with the player you choose to kill. No other text"""

    answer = client.chat(system_prompt= prompt,    
    user_message=scenario)
    print("RESPONSE:", answer)
    #wanted to see if they still respons as they should, even with a lot of information in the system prompt
    
    

INFO:httpx:HTTP Request: POST https://chat-ai.academiccloud.de/v1/chat/completions "HTTP/1.1 200 OK"


RESPONSE: [2]


In [54]:
from openai import OpenAI, OpenAIError
import itertools
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)


class RotatingOpenAIClient:
    def __init__(self, api_keys, base_url, model, system_prompt="You are a helpful assistant"):
        if not api_keys:
            raise ValueError("At least one API key is required")
        self.api_keys = list(api_keys)
        self.base_url = base_url
        self.model = model
        self.system_prompt = system_prompt
        self._key_cycle = itertools.cycle(self.api_keys)
        self._current_key = next(self._key_cycle)
        self._client = self._build_client(self._current_key)

    def _build_client(self, api_key):
        return OpenAI(api_key=api_key, base_url=self.base_url)

    def _rotate_key(self):
        self._current_key = next(self._key_cycle)
        self._client = self._build_client(self._current_key)
        logger.info(f"Rotated to key ending in ...{self._current_key[-4:]}")

    def chat(self, user_message, system_prompt=None, max_retries=None, **kwargs):
        """
        Send a chat message, rotating through keys on failure.
        Tries each key once by default before giving up.
        """
        max_retries = max_retries or len(self.api_keys)
        last_error = None

        for attempt in range(max_retries):
            try:
                response = self._client.chat.completions.create(
                    model=self.model,
                    messages=[
                        {"role": "system", "content": system_prompt or self.system_prompt},
                        {"role": "user", "content": user_message},
                    ],
                    **kwargs,
                )
                return response.choices[0].message.content
            except OpenAIError as e:
                last_error = e
                logger.warning(
                    f"Key ...{self._current_key[-4:]} failed (attempt {attempt + 1}/{max_retries}): {e}"
                )
                self._rotate_key()

        raise RuntimeError(f"All {max_retries} attempts failed. Last error: {last_error}")


# --- Usage ---
if __name__ == "__main__":
    api_keys = [
        "bc315679fc9b7675265b194d420449a0",
        "de905758b4bf5da0bc26ec7555c2cd92"
    ]

    client = RotatingOpenAIClient(
        api_keys=api_keys,
        base_url="https://chat-ai.academiccloud.de/v1",
        model="llama-3.3-70b-instruct",
    )

    scenario = r"""\n[GAME] Welcome to Secret Mafia! You are Player 2.\nYour role: Doctor\nTeam: Village\nDescription: Protect one player each night from Mafia elimination.\n\nPlayers: Player 0, Player 1, Player 2, Player 3, Player 4, Player 5\n\nDuring DAY phase: Speak freely and vote.\nDuring NIGHT phase: '[Player X]' to protect a player.\nWin by identifying and eliminating all Mafia members.\n\n[GAME] Night phase - choose one player to protect: [0], [3], [4], [5]"""
    prompt = """You are playing the game Secret Mafia.
    Your role: Doctor
    Description: Protect one player each night from Mafia elimination.
    Choose a player in the following format: [X].
    Only respond with the player you choose to protect. No other text"""

    answer = client.chat(system_prompt= prompt,    
    user_message=scenario)
    print("RESPONSE:", answer)
    #wanted to see if they still respons as they should, even with a lot of information in the system prompt
    

INFO:httpx:HTTP Request: POST https://chat-ai.academiccloud.de/v1/chat/completions "HTTP/1.1 200 OK"


RESPONSE: [0]


In [55]:
from openai import OpenAI, OpenAIError
import itertools
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)


class RotatingOpenAIClient:
    def __init__(self, api_keys, base_url, model, system_prompt="You are a helpful assistant"):
        if not api_keys:
            raise ValueError("At least one API key is required")
        self.api_keys = list(api_keys)
        self.base_url = base_url
        self.model = model
        self.system_prompt = system_prompt
        self._key_cycle = itertools.cycle(self.api_keys)
        self._current_key = next(self._key_cycle)
        self._client = self._build_client(self._current_key)

    def _build_client(self, api_key):
        return OpenAI(api_key=api_key, base_url=self.base_url)

    def _rotate_key(self):
        self._current_key = next(self._key_cycle)
        self._client = self._build_client(self._current_key)
        logger.info(f"Rotated to key ending in ...{self._current_key[-4:]}")

    def chat(self, user_message, system_prompt=None, max_retries=None, **kwargs):
        """
        Send a chat message, rotating through keys on failure.
        Tries each key once by default before giving up.
        """
        max_retries = max_retries or len(self.api_keys)
        last_error = None

        for attempt in range(max_retries):
            try:
                response = self._client.chat.completions.create(
                    model=self.model,
                    messages=[
                        {"role": "system", "content": system_prompt or self.system_prompt},
                        {"role": "user", "content": user_message},
                    ],
                    **kwargs,
                )
                return response.choices[0].message.content
            except OpenAIError as e:
                last_error = e
                logger.warning(
                    f"Key ...{self._current_key[-4:]} failed (attempt {attempt + 1}/{max_retries}): {e}"
                )
                self._rotate_key()

        raise RuntimeError(f"All {max_retries} attempts failed. Last error: {last_error}")


# --- Usage ---
if __name__ == "__main__":
    api_keys = [
        "bc315679fc9b7675265b194d420449a0",
        "de905758b4bf5da0bc26ec7555c2cd92"
    ]

    client = RotatingOpenAIClient(
        api_keys=api_keys,
        base_url="https://chat-ai.academiccloud.de/v1",
        model="llama-3.3-70b-instruct",
    )

    scenario = r"""\n[GAME] Welcome to Secret Mafia! You are Player 5.\nYour role: Detective\nTeam: Village\nDescription: Investigate players to find Mafia members.\n\nPlayers: Player 0, Player 1, Player 2, Player 3, Player 4, Player 5\n\nDuring DAY phase: Speak freely and vote.\nDuring NIGHT phase: '[Player X]' to investigate.\nYou'll learn immediately if the target is Mafia.\nWin by identifying and eliminating all Mafia members.\n\n[GAME] Night phase - choose one player to investigate: [0], [1], [2], [3], [4]"""
    prompt = """You are playing the game Secret Mafia.
    Your role: Detective
    Description: Investigate players to find Mafia members.
    Choose a player in the following format: [X].
    Only respond with the player you choose to investigate. No other text"""

    answer = client.chat(system_prompt= prompt,    
    user_message=scenario)
    print("RESPONSE:", answer)
    #wanted to see if they still respons as they should, even with a lot of information in the system prompt
    

INFO:httpx:HTTP Request: POST https://chat-ai.academiccloud.de/v1/chat/completions "HTTP/1.1 200 OK"


RESPONSE: [2]
